# Nanbeige4.1-3B: Comprehensive Tutorial for AI Engineers

**A 3B-parameter model that rivals 30B+ models in reasoning, code generation, and agentic tool use.**

---

## Table of Contents

1. [Introduction & Architecture Overview](#1-introduction)
2. [Environment Setup](#2-setup)
3. [Model Loading & Basic Inference](#3-loading)
4. [Reasoning & Chain-of-Thought](#4-reasoning)
5. [Code Generation](#5-code-generation)
6. [Agentic Tool Use](#6-tool-use)
7. [Multi-Turn Conversations](#7-multi-turn)
8. [Benchmarking & Evaluation](#8-benchmarks)
9. [Memory-Efficient Inference (Quantization)](#9-quantization)
10. [vLLM Deployment](#10-vllm)
11. [Temperature & Sampling Experiments](#11-sampling)
12. [Practical Tips & Known Limitations](#12-tips)

---

## 1. Introduction & Architecture Overview <a id='1-introduction'></a>

### What is Nanbeige4.1-3B?

Nanbeige4.1-3B is a **3-billion-parameter open-source small language model (SLM)** built on top of `Nanbeige4-3B-Base`. It integrates:

- **Agentic tool use** — up to 600 tool-call turns in a single session
- **Code generation** — surpassing models 10x its size on LiveCodeBench
- **Mathematical & scientific reasoning** — competitive with Qwen3-32B on GPQA and AIME

### Key Training Techniques

| Technique | Description |
|---|---|
| **Point-wise Reward Modeling** | Scores individual responses for quality on a numeric scale |
| **Pair-wise Reward Modeling** | Compares response pairs to learn relative preferences |
| **Complexity-Aware RL** | Adjusts reinforcement learning rewards based on task difficulty |
| **Supervised Fine-Tuning (SFT)** | Initial instruction tuning on curated datasets |
| **Reinforcement Learning (RL)** | GRPO-based optimization after SFT |

### Benchmark Highlights

| Benchmark | Nanbeige4.1-3B | Qwen3-30B-A3B | Qwen3-32B |
|---|---|---|---|
| LiveCodeBench | **76.9** | 66.0 | 68.3 |
| GPQA-Diamond | **82.2** | 73.4 | 68.7 |
| AIME 2024 (avg@8) | **90.4** | 89.2 | 81.4 |
| AIME 2025 (avg@8) | **85.6** | 85.0 | 72.9 |
| xBench-DeepSearch-05 | **75.0** | — | — |
| Arena-Hard V2 | **73.8** | — | — |
| BFCL-V4 (Tool Use) | **53.8** | 48.6 | — |
| GAIA | **69.9** | — | — |

### Architecture

- **Base architecture**: LLaMA-style decoder-only transformer
- **Parameters**: ~3B (active) / ~4B (total with embeddings)
- **Context window**: 32,768 tokens
- **License**: Apache 2.0
- **HuggingFace**: `Nanbeige/Nanbeige4.1-3B`

## 2. Environment Setup <a id='2-setup'></a>

### Using uv (recommended)

```bash
# Install uv if not already installed
curl -LsSf https://astral.sh/uv/install.sh | sh

# Create and activate environment
cd tutorials/nanbeige-ai-tutorial
uv sync

# For vLLM support
uv sync --extra vllm

# For quantization support
uv sync --extra quantize

# Run the notebook
uv run jupyter notebook nanbeige_tutorial.ipynb

# Run the Gradio app
uv run python app.py
```

### GPU Requirements

| Mode | VRAM Required | Notes |
|---|---|---|
| FP16 | ~6-8 GB | Full precision, best quality |
| INT8 | ~3-4 GB | Good quality, lower memory |
| INT4 | ~2-3 GB | Acceptable quality, minimal memory |
| CPU-only | 8+ GB RAM | Slow but functional for testing |

In [ ]:
# Cell 1: Verify environment
import sys
import torch

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("Running on CPU — inference will be slower but functional")

## 3. Model Loading & Basic Inference <a id='3-loading'></a>

We demonstrate three loading strategies:
1. **AutoModelForCausalLM** — standard HuggingFace approach
2. **Pipeline** — simplified high-level API
3. **Device-aware** — automatic GPU/CPU selection

In [ ]:
# Cell 2: Load model with AutoModelForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MODEL_ID = "Nanbeige/Nanbeige4.1-3B"

# Detect device
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print(f"Loading {MODEL_ID} on {device} with {dtype}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto" if device == "cuda" else None,
    trust_remote_code=True,
)

if device == "cpu":
    model = model.to(device)

print(f"Model loaded. Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")

In [ ]:
# Cell 3: Helper function for generation
import time


def generate_response(
    messages,
    max_new_tokens=2048,
    temperature=0.6,
    top_p=0.95,
    do_sample=True,
    show_thinking=True,
):
    """Generate a response from the model given a list of messages.

    Args:
        messages: List of dicts with 'role' and 'content' keys.
        max_new_tokens: Maximum tokens to generate.
        temperature: Sampling temperature (0.5-0.6 recommended for creativity).
        top_p: Nucleus sampling threshold.
        do_sample: Whether to sample (False = greedy decoding).
        show_thinking: Whether to display <think> blocks.

    Returns:
        The generated text response.
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if do_sample else None,
            top_p=top_p if do_sample else None,
            do_sample=do_sample,
        )
    elapsed = time.time() - start

    # Decode only the new tokens
    new_tokens = outputs[0][inputs["input_ids"].shape[1] :]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True)

    tokens_generated = len(new_tokens)
    tokens_per_sec = tokens_generated / elapsed if elapsed > 0 else 0

    # Separate thinking from final answer
    thinking = ""
    answer = response
    if "<think>" in response and "</think>" in response:
        think_start = response.index("<think>") + len("<think>")
        think_end = response.index("</think>")
        thinking = response[think_start:think_end].strip()
        answer = response[think_end + len("</think>") :].strip()

    print(f"--- Generated {tokens_generated} tokens in {elapsed:.1f}s ({tokens_per_sec:.1f} tok/s) ---")
    if thinking and show_thinking:
        print(f"\n[Thinking]\n{thinking[:500]}{'...' if len(thinking) > 500 else ''}")
    print(f"\n[Response]\n{answer}")

    return {"response": answer, "thinking": thinking, "tokens": tokens_generated, "time": elapsed}

In [ ]:
# Cell 4: Basic inference — simple question
result = generate_response(
    [{"role": "user", "content": "What is the capital of France? Answer in one sentence."}],
    max_new_tokens=128,
    temperature=0.6,
)

## 4. Reasoning & Chain-of-Thought <a id='4-reasoning'></a>

Nanbeige4.1-3B uses `<think>...</think>` blocks for internal reasoning before answering.
This is the key to its strong AIME and GPQA scores — the model "thinks" before responding.

### Key insight: Complexity-Aware RL
The model was trained with reinforcement learning that **adjusts rewards based on problem difficulty**.
Harder problems get more reward signal, pushing the model to develop deeper reasoning chains.

In [ ]:
# Cell 5: Math reasoning (AIME-style problem)
math_prompt = """Solve this step by step:

Find all positive integers n such that n^2 + 3n + 5 is divisible by 121."""

result = generate_response(
    [{"role": "user", "content": math_prompt}],
    max_new_tokens=4096,
    temperature=0.6,
)

In [ ]:
# Cell 6: Scientific reasoning (GPQA-style)
science_prompt = """A researcher observes that a certain enzyme's activity increases with
temperature up to 40°C, then drops sharply. However, when the same enzyme is bound to a
thermostable protein scaffold, it remains active up to 70°C.

Explain the molecular mechanism behind this observation, including:
1. Why the unmodified enzyme denatures at 40°C
2. How the protein scaffold provides thermostability
3. What types of non-covalent interactions are likely involved"""

result = generate_response(
    [{"role": "user", "content": science_prompt}],
    max_new_tokens=4096,
    temperature=0.6,
)

In [ ]:
# Cell 7: Logic puzzle
logic_prompt = """Three people — Alice, Bob, and Charlie — each have a different favorite color
(red, blue, green) and a different pet (cat, dog, fish).

Clues:
1. Alice does not like red.
2. The person who likes blue has a dog.
3. Charlie has a fish.
4. Bob does not like green.

Determine each person's favorite color and pet."""

result = generate_response(
    [{"role": "user", "content": logic_prompt}],
    max_new_tokens=2048,
    temperature=0.6,
)

## 5. Code Generation <a id='5-code-generation'></a>

Nanbeige4.1-3B scores **76.9%** on LiveCodeBench — comparable to DeepSeek R1 (670B parameters).
This makes it one of the most parameter-efficient code generators available.

### Best practices for code generation:
- Use `temperature=0.6` for balanced creativity/correctness
- Provide clear specifications in the prompt
- For extended coding tasks, be aware of occasional bugs (known limitation)

In [ ]:
# Cell 8: Algorithm implementation
code_prompt = """Write a Python implementation of a Trie (prefix tree) data structure with the
following methods:
- insert(word: str) -> None
- search(word: str) -> bool
- starts_with(prefix: str) -> list[str]
- delete(word: str) -> bool

Include type hints and a brief docstring for each method."""

result = generate_response(
    [{"role": "user", "content": code_prompt}],
    max_new_tokens=4096,
    temperature=0.6,
)

In [ ]:
# Cell 9: Data processing pipeline
data_prompt = """Write a Python function that processes a CSV file with columns:
'timestamp', 'sensor_id', 'temperature', 'humidity'.

The function should:
1. Read the CSV using pandas
2. Handle missing values (forward-fill then backward-fill)
3. Detect anomalies (values > 3 standard deviations from the rolling mean)
4. Return a summary DataFrame with per-sensor statistics

Use type hints and handle edge cases."""

result = generate_response(
    [{"role": "user", "content": data_prompt}],
    max_new_tokens=4096,
    temperature=0.6,
)

In [ ]:
# Cell 10: Debug and fix code
debug_prompt = """The following Python code has bugs. Find and fix all issues:

```python
def merge_sorted_lists(list1, list2):
    result = []
    i = j = 0
    while i < len(list1) and j < len(list2):
        if list1[i] <= list2[j]:
            result.append(list1[i])
            i += 1
        else:
            result.append(list2[j])
            i += 1  # Bug 1
    result.extend(list1[i:])
    # Bug 2: missing list2 remainder
    return result

def binary_search(arr, target):
    left, right = 0, len(arr)
    while left < right:  # Bug 3: should be <=
        mid = (left + right) / 2  # Bug 4: integer division
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid  # Bug 5: should be mid + 1
        else:
            right = mid - 1
    return -1
```

Explain each bug and provide the corrected code."""

result = generate_response(
    [{"role": "user", "content": debug_prompt}],
    max_new_tokens=4096,
    temperature=0.6,
)

## 6. Agentic Tool Use <a id='6-tool-use'></a>

One of Nanbeige4.1-3B's standout features is **agentic tool use** — the model can invoke
external tools (functions) in a structured JSON format and handle up to **600 tool-call turns**
in a single conversation.

### How it works:
1. You define tools as JSON schemas in the system prompt
2. The model generates structured `tool_call` outputs when it needs external data
3. You execute the tool and feed results back
4. The model incorporates results into its reasoning

### Benchmark: BFCL-V4 = 53.8 (5.2 points above Qwen3-30B-A3B)

In [ ]:
# Cell 11: Define tools for the model
import json

# Define tool schemas
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather for a location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "City name, e.g. 'San Francisco, CA'",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature unit",
                    },
                },
                "required": ["location"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate",
            "description": "Evaluate a mathematical expression",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "Mathematical expression to evaluate, e.g. '2 + 3 * 4'",
                    }
                },
                "required": ["expression"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_database",
            "description": "Search a product database by query",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Search query"},
                    "category": {
                        "type": "string",
                        "enum": ["electronics", "books", "clothing"],
                        "description": "Product category",
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results",
                        "default": 5,
                    },
                },
                "required": ["query"],
            },
        },
    },
]

print(f"Defined {len(TOOLS)} tools:")
for tool in TOOLS:
    print(f"  - {tool['function']['name']}: {tool['function']['description']}")

In [ ]:
# Cell 12: Tool-calling conversation

# Build system prompt with tool definitions
tool_system_prompt = """You are a helpful assistant with access to the following tools:

""" + json.dumps(TOOLS, indent=2) + """

When you need to use a tool, respond with a JSON block:
{"tool_call": {"name": "tool_name", "arguments": {"arg1": "value1"}}}

After receiving tool results, incorporate them into your response."""

messages = [
    {"role": "system", "content": tool_system_prompt},
    {
        "role": "user",
        "content": "What's the weather like in Tokyo? Also, calculate 15% tip on a $85.50 bill.",
    },
]

result = generate_response(
    messages, max_new_tokens=2048, temperature=0.6
)

In [ ]:
# Cell 13: Simulated tool execution loop


def simulate_tool_call(name, arguments):
    """Simulate tool execution with mock responses."""
    mock_responses = {
        "get_weather": lambda args: {
            "location": args.get("location", "Unknown"),
            "temperature": 22,
            "unit": args.get("unit", "celsius"),
            "condition": "Partly cloudy",
            "humidity": 65,
        },
        "calculate": lambda args: {
            "expression": args.get("expression", ""),
            "result": eval(args.get("expression", "0")),  # NOTE: only for demo
        },
        "search_database": lambda args: {
            "results": [
                {"name": "Sample Product", "price": 29.99, "rating": 4.5},
                {"name": "Another Product", "price": 49.99, "rating": 4.2},
            ],
            "total": 2,
        },
    }
    handler = mock_responses.get(name)
    if handler:
        return handler(arguments)
    return {"error": f"Unknown tool: {name}"}


def run_agent_loop(user_query, max_turns=5):
    """Run a multi-turn agent loop with tool calling."""
    messages = [
        {"role": "system", "content": tool_system_prompt},
        {"role": "user", "content": user_query},
    ]

    for turn in range(max_turns):
        print(f"\n{'='*60}")
        print(f"Turn {turn + 1}")
        print(f"{'='*60}")

        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs, max_new_tokens=2048, temperature=0.6, top_p=0.95, do_sample=True
            )

        new_tokens = outputs[0][inputs["input_ids"].shape[1] :]
        response = tokenizer.decode(new_tokens, skip_special_tokens=True)

        # Remove thinking blocks for cleaner display
        clean_response = response
        if "<think>" in response and "</think>" in response:
            think_end = response.index("</think>") + len("</think>")
            clean_response = response[think_end:].strip()

        print(f"Assistant: {clean_response[:500]}")

        # Check for tool calls in the response
        try:
            # Try to extract JSON tool call
            if '"tool_call"' in clean_response:
                json_start = clean_response.index("{")
                json_end = clean_response.rindex("}") + 1
                tool_data = json.loads(clean_response[json_start:json_end])
                tool_call = tool_data.get("tool_call", {})

                name = tool_call.get("name", "")
                args = tool_call.get("arguments", {})

                print(f"\n[Tool Call] {name}({json.dumps(args)})")
                tool_result = simulate_tool_call(name, args)
                print(f"[Tool Result] {json.dumps(tool_result)}")

                messages.append({"role": "assistant", "content": response})
                messages.append(
                    {
                        "role": "user",
                        "content": f"Tool result: {json.dumps(tool_result)}",
                    }
                )
                continue
        except (json.JSONDecodeError, ValueError):
            pass

        # No tool call — final answer
        print("\n[Agent completed — no more tool calls needed]")
        return clean_response

    print("\n[Max turns reached]")
    return clean_response


# Run the agent
final_answer = run_agent_loop(
    "What's the weather in San Francisco? Then search for umbrellas if it's rainy."
)

## 7. Multi-Turn Conversations <a id='7-multi-turn'></a>

The model handles extended multi-turn conversations well, maintaining context over
many exchanges. This is critical for agentic workflows where the model may need
to perform dozens of tool calls before reaching a conclusion.

In [ ]:
# Cell 14: Multi-turn conversation with context tracking

conversation = [
    {"role": "system", "content": "You are a helpful coding tutor. Be concise."},
]

turns = [
    "Explain what a Python decorator is in 2 sentences.",
    "Now show me a simple example of a timing decorator.",
    "How would I modify it to also log the function arguments?",
    "Can this decorator be used on async functions? If not, how to adapt it?",
]

for i, user_msg in enumerate(turns, 1):
    print(f"\n{'='*60}")
    print(f"Turn {i}: {user_msg}")
    print(f"{'='*60}")

    conversation.append({"role": "user", "content": user_msg})
    result = generate_response(
        conversation, max_new_tokens=1024, temperature=0.6, show_thinking=False
    )
    conversation.append({"role": "assistant", "content": result["response"]})

## 8. Benchmarking & Evaluation <a id='8-benchmarks'></a>

Let's run our own mini-benchmarks to evaluate the model on different task types.
This gives you a practical feel for model capabilities beyond published numbers.

In [ ]:
# Cell 15: Mini-benchmark suite
import pandas as pd

BENCHMARK_TASKS = [
    {
        "category": "Math",
        "prompt": "What is the sum of all prime numbers less than 50?",
        "expected": "328",
    },
    {
        "category": "Math",
        "prompt": "Solve: If f(x) = 3x^2 - 2x + 1, what is f(4)?",
        "expected": "41",
    },
    {
        "category": "Code",
        "prompt": "Write a Python one-liner to flatten a nested list [[1,2],[3,[4,5]],6] into [1,2,3,4,5,6]. Just the code, no explanation.",
        "expected": "flatten",
    },
    {
        "category": "Code",
        "prompt": "What does this Python expression evaluate to: sorted(set('banana'))?",
        "expected": "['a', 'b', 'n']",
    },
    {
        "category": "Reasoning",
        "prompt": "If all roses are flowers and some flowers fade quickly, can we conclude that some roses fade quickly? Answer yes or no with a brief explanation.",
        "expected": "no",
    },
    {
        "category": "Reasoning",
        "prompt": "A bat and a ball cost $1.10 total. The bat costs $1.00 more than the ball. How much does the ball cost?",
        "expected": "$0.05",
    },
    {
        "category": "Science",
        "prompt": "What is the primary function of mitochondria in a cell? Answer in one sentence.",
        "expected": "energy",
    },
    {
        "category": "Tool Use",
        "prompt": 'Given this tool: {"name": "get_price", "params": {"symbol": "string"}}. Generate a valid JSON tool call to get the price of AAPL stock.',
        "expected": 'AAPL',
    },
]

results = []
for task in BENCHMARK_TASKS:
    print(f"\n[{task['category']}] {task['prompt'][:60]}...")
    result = generate_response(
        [{"role": "user", "content": task["prompt"]}],
        max_new_tokens=512,
        temperature=0.6,
        show_thinking=False,
    )
    contains_expected = task["expected"].lower() in result["response"].lower()
    results.append(
        {
            "Category": task["category"],
            "Contains Expected": contains_expected,
            "Tokens": result["tokens"],
            "Time (s)": round(result["time"], 1),
        }
    )

df = pd.DataFrame(results)
print("\n" + "=" * 60)
print("BENCHMARK RESULTS")
print("=" * 60)
print(df.to_string(index=False))
print(f"\nOverall accuracy: {df['Contains Expected'].mean():.1%}")
print(f"Average tokens: {df['Tokens'].mean():.0f}")
print(f"Average time: {df['Time (s)'].mean():.1f}s")

## 9. Memory-Efficient Inference (Quantization) <a id='9-quantization'></a>

For resource-constrained environments, quantization reduces memory usage significantly.

### Community note on GGUF
As of the time of writing, **official GGUF quantization is not provided** by Nanbeige.
Community-created GGUF versions exist (e.g., `DevQuasar/Nanbeige.Nanbeige4.1-3B-GGUF`)
and Ollama models like `fauxpaslife/nanbeige4.1-python-deepthink:3b-q8` are available.

Below we show `bitsandbytes` INT8/INT4 loading which works natively with HuggingFace.

In [ ]:
# Cell 16: INT8 quantized loading (requires: uv sync --extra quantize)
# Uncomment to run — requires bitsandbytes and GPU

"""
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# INT8 quantization config
quantization_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
)

# INT4 quantization config (even smaller)
quantization_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Choose one:
config = quantization_config_8bit  # or quantization_config_4bit

model_quantized = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=config,
    device_map="auto",
    trust_remote_code=True,
)

# Check memory savings
mem_bytes = sum(p.nelement() * p.element_size() for p in model_quantized.parameters())
print(f"Quantized model memory: {mem_bytes / 1e9:.2f} GB")
"""
print("Quantization examples above — uncomment to run with GPU + bitsandbytes")

## 10. vLLM Deployment <a id='10-vllm'></a>

For production serving, [vLLM](https://docs.vllm.ai) provides high-throughput inference
with an OpenAI-compatible API.

### Setup
```bash
# Install with vLLM extras
uv sync --extra vllm

# Start the server
uv run vllm serve Nanbeige/Nanbeige4.1-3B \
    --trust-remote-code \
    --max-model-len 32768 \
    --enable-reasoning \
    --reasoning-parser deepseek_r1
```

### Key flags
- `--enable-reasoning --reasoning-parser deepseek_r1`: Separates `<think>` blocks into `reasoning_content`
- `--max-model-len 32768`: Full context window
- `--gpu-memory-utilization 0.9`: Control VRAM usage

In [ ]:
# Cell 17: vLLM client example (requires running vLLM server)
# Start the server first: uv run vllm serve Nanbeige/Nanbeige4.1-3B --trust-remote-code

"""
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")

# Basic chat completion
response = client.chat.completions.create(
    model="Nanbeige/Nanbeige4.1-3B",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Explain the CAP theorem in distributed systems."},
    ],
    temperature=0.6,
    top_p=0.95,
    max_tokens=2048,
)

print(response.choices[0].message.content)

# With tool calling (vLLM supports OpenAI-compatible function calling)
response_tools = client.chat.completions.create(
    model="Nanbeige/Nanbeige4.1-3B",
    messages=[{"role": "user", "content": "What's the weather in Paris?"}],
    tools=[
        {
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get weather for a city",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name"}
                    },
                    "required": ["city"],
                },
            },
        }
    ],
    tool_choice="auto",
    temperature=0.6,
)

if response_tools.choices[0].message.tool_calls:
    for tc in response_tools.choices[0].message.tool_calls:
        print(f"Tool: {tc.function.name}, Args: {tc.function.arguments}")
"""
print("vLLM client example above — uncomment after starting the vLLM server")

## 11. Temperature & Sampling Experiments <a id='11-sampling'></a>

Community feedback indicates that **temperatures of 0.5-0.6 yield the best creative results**.
Let's experiment with different settings to verify this.

In [ ]:
# Cell 18: Temperature comparison experiment

test_prompt = "Write a creative opening line for a sci-fi novel set on Mars."

temperatures = [0.3, 0.5, 0.6, 0.8, 1.0]
temp_results = []

for temp in temperatures:
    print(f"\n--- Temperature: {temp} ---")
    result = generate_response(
        [{"role": "user", "content": test_prompt}],
        max_new_tokens=256,
        temperature=temp,
        show_thinking=False,
    )
    temp_results.append({"temperature": temp, "response": result["response"][:200]})

print("\n" + "=" * 60)
print("TEMPERATURE COMPARISON SUMMARY")
print("=" * 60)
for r in temp_results:
    print(f"\nT={r['temperature']}: {r['response'][:150]}...")

In [ ]:
# Cell 19: Greedy vs. sampling comparison on a factual task

factual_prompt = "List the first 10 elements of the periodic table with their atomic numbers."

print("=== Greedy Decoding (temperature=None, do_sample=False) ===")
greedy_result = generate_response(
    [{"role": "user", "content": factual_prompt}],
    max_new_tokens=512,
    do_sample=False,
    show_thinking=False,
)

print("\n=== Sampling (temperature=0.6, top_p=0.95) ===")
sampled_result = generate_response(
    [{"role": "user", "content": factual_prompt}],
    max_new_tokens=512,
    temperature=0.6,
    show_thinking=False,
)

## 12. Practical Tips & Known Limitations <a id='12-tips'></a>

### Recommended Settings

| Use Case | Temperature | top_p | max_tokens | Notes |
|---|---|---|---|---|
| Code generation | 0.6 | 0.95 | 4096 | Best balance of correctness and variety |
| Math/reasoning | 0.6 | 0.95 | 4096 | Let the model think — longer outputs are better |
| Creative writing | 0.5-0.6 | 0.95 | 2048 | Community reports best creativity here |
| Factual Q&A | 0.0 (greedy) | 1.0 | 512 | Deterministic for accuracy |
| Tool calling | 0.6 | 0.95 | 2048 | Structured output needs some flexibility |
| Evaluation (avg@k) | 0.6 | 0.95 | 32768 | Match official benchmark settings |

### Known Limitations

1. **No official GGUF quantization** — Community GGUF versions exist but are unofficial
2. **Extended coding sessions** — Occasional bugs in very long multi-step coding tasks
3. **Context window** — 32K tokens is generous but not infinite; long tool-call chains can approach limits
4. **Benchmark concerns** — Authors may use benchmark sets for validation/checkpoint selection
5. **Hardware requirements** — Still needs ~6-8 GB VRAM for FP16; INT4 brings this to ~2-3 GB

### Tips for Production

- **Use vLLM** for production serving — significant throughput improvement over raw transformers
- **Enable reasoning parsing** (`--enable-reasoning --reasoning-parser deepseek_r1`) to separate thinking from answers
- **Monitor token usage** — the model can generate long `<think>` blocks; set appropriate `max_tokens`
- **Tool call validation** — Always validate the JSON structure of tool calls before execution
- **Fallback strategy** — For critical applications, consider a larger model as fallback when confidence is low

In [ ]:
# Cell 20: Cleanup
import gc

# Free GPU memory
if "model" in dir():
    del model
if "tokenizer" in dir():
    del tokenizer
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory freed. Current usage: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
else:
    print("Cleanup complete.")

print("\nTutorial complete! Next steps:")
print("  1. Try the interactive Gradio app: uv run python app.py")
print("  2. Deploy with vLLM: uv run vllm serve Nanbeige/Nanbeige4.1-3B --trust-remote-code")
print("  3. Explore community GGUF: DevQuasar/Nanbeige.Nanbeige4.1-3B-GGUF")